# logsumexp-cross-entropy — worked example 3: Closed-form CE gradient (softmax minus onehot)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `logsumexp-cross-entropy`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The gradient of mean-reduced cross-entropy w.r.t. the logits is `(softmax(logits) - onehot(target)) / B`. This one-step identity avoids calling backward and is the standard analytic result used in hand-rolled trainers.

## Worked solution

We compute the CE gradient in closed form and validate it with autograd.

1. **Softmax.** `probs = softmax(logits, dim=-1)`, shape `(B, C)`.
2. **One-hot target.** `onehot = F.one_hot(target, C).float()`, same shape, with a 1 at each correct class.
3. **Gradient.** `(probs - onehot) / B`. Dividing by the batch size accounts for the mean reduction in the forward loss.
4. **Witness.** We let autograd differentiate the same mean cross-entropy and confirm the closed form matches.

The demo builds random logits, computes the closed form, and prints whether it matches `torch.autograd`.

In [ ]:
import torch as t
import torch.nn.functional as F

t.manual_seed(2)

def ce_grad(logits, target):
    B, C = logits.shape
    probs = t.softmax(logits, dim=-1)
    onehot = F.one_hot(target, num_classes=C).to(probs.dtype)
    return (probs - onehot) / B

logits = t.randn(4, 5)
target = t.tensor([0, 2, 4, 1])
grad_cf = ce_grad(logits, target)

lv = logits.clone().detach().requires_grad_(True)
lse = t.logsumexp(lv, dim=-1)
loss = (lse - lv[t.arange(4), target]).mean()
loss.backward()
print('grad shape:', tuple(grad_cf.shape))
print('matches autograd:', t.allclose(grad_cf, lv.grad, atol=1e-6))